# MCP en porties schalen

## Waarom een aparte tool?
Een taalmodel kan een vraag begrijpen, maar hoort hoeveelheden niet uit het hoofd te berekenen. De MCP-server biedt `scale_recipe` aan met een naam, beschrijving en gestructureerde invoer. De factor is gewenste porties / oorspronkelijke porties. Negatieve hoeveelheden en niet-positieve porties worden geweigerd. De uitkomst wordt op drie decimalen afgerond; dit kan voor kleine hoeveelheden betekenisvol zijn.

De server werkt via standaardinvoer en -uitvoer als afzonderlijk proces. MCP verzorgt de verbinding; de agent blijft verantwoordelijk voor het kiezen van de tool en controleren van de argumenten. Deze functie voert geen omzetting tussen gewicht en volume uit: daarvoor is ingrediëntspecifieke dichtheid nodig.


## Uitvoerbare implementatie
Voer deze cel uit vanuit de projectmap om de uitvoerbare versie bij te werken.


In [1]:
%%writefile chef/mcp_server.py
"""Local MCP server with deterministic recipe scaling."""
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("culinary-tools")

@mcp.tool()
def scale_recipe(ingredients: list[dict], from_servings: float, to_servings: float) -> list[dict]:
    """Scale ingredient quantities; each ingredient has name, quantity and unit."""
    if from_servings <= 0 or to_servings <= 0:
        raise ValueError("Serving counts must be positive")
    result = []
    for item in ingredients:
        amount = float(item["quantity"])
        if amount < 0:
            raise ValueError("Ingredient quantities cannot be negative")
        result.append({**item, "quantity": round(amount * to_servings / from_servings, 3)})
    return result

if __name__ == "__main__":
    mcp.run(transport="stdio")


Overwriting chef/mcp_server.py
